# 05 - Polar Accesslink Import

Alternativer Daten-Import **direkt via Polar REST API** - kein manueller ZIP-Export noetig.

## Voraussetzungen

1. **Polar Accesslink App registrieren** (einmalig):  
   -> [admin.polaraccesslink.com](https://admin.polaraccesslink.com)  
   -> "Create new client" mit Redirect URI: `http://localhost:5678/callback`

2. **GitHub Codespaces Secrets** setzen:  
   -> `POLAR_CLIENT_ID` und `POLAR_CLIENT_SECRET` aus dem Developer Portal

3. **Einmalige Autorisierung** (Abschnitt A unten)

---

## Was wird importiert?

| Tabelle | Quelle | Methode |
|---|---|---|
| `polar.activity` | Aktivitaet + Schlaf | Transaction API + Sleep Endpoint |
| `polar.training` | Trainingseinheiten | Exercise Transaction API |
| `polar.physical_info` | Gewicht, Groesse | Physical Info Transaction API |
| `polar.sonstige_daten` | Nightly Recharge | Recharge Endpoint |

**Nicht via API verfuegbar** (weiterhin ZIP-Export noetig):  
- `polar.heartrate` (24/7 HR-Minutendaten)  
- `polar.hrv` (PPI-Rohdaten fuer RMSSD/SDNN)


---
## A - Einmalige Einrichtung (nur beim ersten Mal ausfuehren)

In [18]:
# ╔══════════════════════════════════════════════════════════╗
# ║  EINMALIG - NICHT NOCHMAL AUSFÜHREN (Tokens vorhanden)  ║
# ╚══════════════════════════════════════════════════════════╝
# import sys
# sys.path.insert(0, '../src')
#
# from polar_accesslink import PolarAccesslinkClient
#
# client = PolarAccesslinkClient()
# print(f'Client ID: {client.client_id[:8]}...')
# print(f'Tokens vorhanden: Access={bool(client.access_token)}, Refresh={bool(client.refresh_token)}')
# print(f'User ID: {client.user_id or "noch nicht gesetzt"}')

In [19]:
# ╔══════════════════════════════════════════════════════════╗
# ║  EINMALIG - NICHT NOCHMAL AUSFÜHREN (Tokens vorhanden)  ║
# ╚══════════════════════════════════════════════════════════╝
# NUR EINMALIG ausfuehren - startet OAuth2-Flow
# from pathlib import Path
# if Path('../.polar_tokens.json').exists() or Path('.polar_tokens.json').exists():
#     print('Tokens bereits vorhanden - Zelle ueberspringen')
# else:
#     client.oauth2_setup(
#         redirect_uri='http://localhost:5678/callback',
#         callback_port=5678,
#         timeout=180,
#     )

In [20]:
# ╔══════════════════════════════════════════════════════════╗
# ║  EINMALIG - NICHT NOCHMAL AUSFÜHREN (Tokens vorhanden)  ║
# ╚══════════════════════════════════════════════════════════╝
# NUR EINMALIG ausfuehren - registriert Nutzer bei Accesslink
# info = client.register_user()
# print(info)

In [21]:
# ╔══════════════════════════════════════════════════════════╗
# ║  EINMALIG - NICHT NOCHMAL AUSFÜHREN (Tokens vorhanden)  ║
# ╚══════════════════════════════════════════════════════════╝
# Verbindungstest: Nutzerprofil abrufen
# profil = client.get_user_info()
# print('Polar-Profil:')
# for k, v in profil.items():
#     print(f'  {k}: {v}')

---
## B - Regulaerer Import (taeglich ausfuehren)

In [22]:
import sys
sys.path.insert(0, '../src')

from polar_accesslink import AccesslinkUpdater

# Importiert alle verfuegbaren neuen Daten und schreibt sie in Databricks
# sleep_tage / recharge_tage: wie weit zurueck Schlafdaten geholt werden
with AccesslinkUpdater() as updater:
    bericht = updater.import_alle(
        sleep_tage=28,
        recharge_tage=28,
    )

print('\nBericht:', bericht)



🌐 Polar Accesslink Import
   2026-03-28 10:33:33

📊 Aktivität & Schlaf...
   ℹ️  Aktivität: Keine neuen Daten seit letztem Import.
   ⏭️  activity: Keine Daten

🏃 Training...
   ℹ️  Training: Keine neuen Daten seit letztem Import.
   ⏭️  training: Keine Daten

⚖️  Physical Info...
   ℹ️  Physical Info: Keine neuen Daten.
   ⏭️  physical_info: Keine Daten

🌙 Nightly Recharge...
   ⏭️  sonstige_daten (nightly_recharge): Keine Daten

✅ Import abgeschlossen in 0.6s
   Gesamt: 0 Datensätze


Bericht: {'aktivitaet': 0, 'training': 0, 'physical_info': 0, 'nightly_recharge': 0, 'dauer_sekunden': 0.6, 'fehler': []}


---
## C - Einzelne Importe (optional)

In [23]:
# Nur Aktivitaets- und Schlafdaten importieren
from datetime import date, timedelta

with AccesslinkUpdater() as updater:
    n = updater.import_aktivitaet(
        sleep_von=date.today() - timedelta(days=14),
        sleep_bis=date.today(),
    )
print(f'{n} Aktivitaetsdatensaetze importiert')



📊 Aktivität & Schlaf...
   ℹ️  Aktivität: Keine neuen Daten seit letztem Import.
   ⏭️  activity: Keine Daten
0 Aktivitaetsdatensaetze importiert


In [24]:
# Nur Trainingsdaten importieren
with AccesslinkUpdater() as updater:
    n = updater.import_training()
print(f'{n} Trainingseinheiten importiert')



🏃 Training...
   ℹ️  Training: Keine neuen Daten seit letztem Import.
   ⏭️  training: Keine Daten
0 Trainingseinheiten importiert


In [25]:
# Schlafdaten abrufen (ohne DB-Schreibzugriff, zur Inspektion)
from polar_accesslink import PolarAccesslinkClient
from datetime import date, timedelta

client = PolarAccesslinkClient()
df_schlaf = client.fetch_sleep(
    von=date.today() - timedelta(days=14),
    bis=date.today(),
)
print(f'{len(df_schlaf)} Schlafnaechte')
df_schlaf


0 Schlafnaechte


""


In [26]:
# Nightly Recharge anzeigen (HRV-avg, ANS-Charge etc.)
import json
import pandas as pd
from polar_accesslink import PolarAccesslinkClient
from datetime import date, timedelta

client = PolarAccesslinkClient()
df_recharge = client.fetch_nightly_recharge(
    von=date.today() - timedelta(days=14),
    bis=date.today(),
)
if not df_recharge.empty:
    expanded = pd.json_normalize(df_recharge['json_data'].apply(json.loads))
    cols = [c for c in ['date', 'ans-charge', 'hrv-avg', 'hrv-diff-from-long-term-avg'] if c in expanded.columns]
    display(expanded[cols].head(14))
else:
    print('Keine Recharge-Daten verfuegbar')


Keine Recharge-Daten verfuegbar


---
## D - Token-Verwaltung

In [27]:
# Token manuell erneuern
from polar_accesslink import PolarAccesslinkClient

client = PolarAccesslinkClient()
ok = client.refresh_access_token()
print(f'Token-Refresh: {"Erfolgreich" if ok else "Fehlgeschlagen (kein Refresh-Token?)"}')


Token-Refresh: Fehlgeschlagen (kein Refresh-Token?)


In [28]:
# Tokenfile-Status pruefen
from pathlib import Path
import json

for p in [Path('.polar_tokens.json'), Path('../.polar_tokens.json')]:
    if p.exists():
        data = json.loads(p.read_text())
        print(f'Token-Datei: {p}')
        print(f'  Access Token : {data.get("access_token", "")[:20]}...')
        print(f'  Refresh Token: {"vorhanden" if data.get("refresh_token") else "fehlt"}')
        print(f'  User ID      : {data.get("user_id")}')
        print(f'  Aktualisiert : {data.get("aktualisiert")}')
        break
else:
    print('Keine Token-Datei gefunden -> Abschnitt A ausfuehren')


Token-Datei: .polar_tokens.json
  Access Token : c2d97b06dab131c1a69b...
  Refresh Token: fehlt
  User ID      : 15900936
  Aktualisiert : 2026-03-28T10:12:42.346265
